# Colab Queue Watch

Thin notebook for tracking queue progress and ETA from the shared Drive workspace.

- Run the snapshot cell for a one-time status + ETA view.
- Run the live watch cell if you want the notebook to refresh until interrupted.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

DRIVE_BASE = Path('/content/drive/MyDrive')
REPO_URL = 'https://github.com/andersvestrum/carbon-aware-recsys.git'
REPO_BRANCH = 'big_man_thing'
REPO_ROOT = DRIVE_BASE / 'carbon-aware-recsys'
DRIVE_ROOT = DRIVE_BASE / 'carbon-aware-recsys-colab'

TAIL_LINES = 12
REFRESH_SECONDS = 60
PARALLELISM = 2  # set to None to auto-infer from active workers

if (REPO_ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    if REPO_ROOT.exists():
        print(f'Removing non-git directory: {REPO_ROOT}')
        shutil.rmtree(REPO_ROOT)
    subprocess.run([
        'git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)
    ], check=True)

print({
    'repo_root': str(REPO_ROOT),
    'drive_root': str(DRIVE_ROOT),
    'repo_branch': REPO_BRANCH,
    'tail_lines': TAIL_LINES,
    'refresh_seconds': REFRESH_SECONDS,
    'parallelism': PARALLELISM,
})


In [ ]:
import subprocess
import sys

watch_script = REPO_ROOT / 'scripts' / '06_colab_watch.py'
cmd = [
    sys.executable,
    str(watch_script),
    '--drive-root', str(DRIVE_ROOT),
    '--tail-lines', str(TAIL_LINES),
]
if PARALLELISM is not None:
    cmd.extend(['--parallelism', str(PARALLELISM)])

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)


In [ ]:
import subprocess
import sys

watch_script = REPO_ROOT / 'scripts' / '06_colab_watch.py'
cmd = [
    sys.executable,
    str(watch_script),
    '--drive-root', str(DRIVE_ROOT),
    '--tail-lines', str(TAIL_LINES),
    '--watch',
    '--refresh-seconds', str(REFRESH_SECONDS),
]
if PARALLELISM is not None:
    cmd.extend(['--parallelism', str(PARALLELISM)])

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)
